# Price matching and finalize

The last stage of the preprocessing pipeline. Three panels come in, one comes out:

| panel | notebook | what it carries |
|---|---|---|
| `f_score_panel.csv` | `f_score_calculation` | 9 raw signals, 9 indicators, `f_score` |
| `book_to_market_panel.csv` | `book_to_market_calculation` | book equity, shares, 31 Dec close, `bm` |
| `trade_turnover_panel.csv` | `trade_turnover_filter` | June window volumes, `turnover`, `tradeable` |

All three key on `(symbol, period)`, where `period` is the **fiscal year**. What they do not
carry is the price you would actually transact at, and that price does not live in the
fiscal year: the portfolio is formed on **30 June of `period` + 1**, once the accounting for
`period` has been published.

    formation date  = 30 June of year F
    matched to      = period F - 1
    entry price     = the last session on or before 30 June F
    exit price      = the last session on or before 30 June F + HOLDING_YEARS

**On or before, never after.** "Nearest to 30 June" would let a 1 July print stand in when
30 June falls on a weekend, and a price you could not have traded at on formation day is
look-ahead, however small. Taking the last prior session costs at most a weekend: 70% of
matched rows land exactly on 30 June and 99% within two days.

In [ ]:
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd

RESULTS = "../data/preprocessing_pipeline_results"
DB = "../cafef.db"

FORMATION_MMDD = "06-30"   # must match trade_turnover_filter.ipynb
FORMATION_LAG_YEARS = 1    # June of year F prices fiscal period F - 1

HOLDING_YEARS = 1          # 30 June F -> 30 June F + 1
MAX_EXIT_LAG_DAYS = 30     # beyond this the exit print is not an exit-day price
DELISTING_RETURN = None    # None = exit at the last traded price; or a number, e.g. -0.30

CUTOFF = 8                 # F-score cutoff, used only for the previews at the end

## Inputs

Read as written, no re-derivation. If any of the three is stale the index check below says
so immediately rather than letting a silent left-join drop rows.

In [ ]:
f_score_panel = pd.read_csv(f"{RESULTS}/f_score_panel.csv").set_index(["symbol", "period"])
bm_panel = pd.read_csv(f"{RESULTS}/book_to_market_panel.csv").set_index(["symbol", "period"])
turnover_panel = pd.read_csv(f"{RESULTS}/trade_turnover_panel.csv").set_index(["symbol", "period"])

pd.DataFrame({
    "rows": [len(f_score_panel), len(bm_panel), len(turnover_panel)],
    "columns": [f_score_panel.shape[1], bm_panel.shape[1], turnover_panel.shape[1]],
}, index=["f_score", "book_to_market", "trade_turnover"])

In [ ]:
# All three are built from the same accounting-clean universe, so their indexes must be
# identical. A difference means one notebook was run against a different vintage of
# f_score_fields_extract_corrected.csv, and every count downstream would be off.
index_check = pd.Series({
    "f_score == bm": f_score_panel.index.equals(bm_panel.index),
    "bm == turnover": bm_panel.index.equals(turnover_panel.index),
    "rows in f_score only": len(f_score_panel.index.difference(bm_panel.index)),
    "rows in bm only": len(bm_panel.index.difference(f_score_panel.index)),
    "rows in turnover only": len(turnover_panel.index.difference(bm_panel.index)),
})
assert index_check["f_score == bm"] and index_check["bm == turnover"], \
    "panels disagree — re-run the notebook that is behind before finalising"
index_check

## The join

`shares_issued` and `has_treasury` are computed in both the BM and the turnover notebook,
from the same `paid_in_capital` of the same row. They are checked against each other and
then kept once — a duplicated column that silently disagrees is worse than no column.

The BM notebook's `close` is the **31 December** close, the price the ratio was measured
against. The formation price added below is a different date and a different purpose, so
the December one is renamed on the way in rather than left to collide.

In [ ]:
assert np.allclose(bm_panel["shares_issued"].fillna(-1),
                   turnover_panel["shares_issued"].fillna(-1)), "shares_issued disagrees"
assert (bm_panel["has_treasury"] == turnover_panel["has_treasury"]).all(), "has_treasury disagrees"

panel = (
    f_score_panel
    .join(bm_panel.rename(columns={"close": "close_dec",
                                   "close_date": "close_dec_date",
                                   "price_stale": "close_dec_stale"}))
    .join(turnover_panel.drop(columns=["shares_issued", "has_treasury"]))
)

print(f"{panel.shape[0]} rows x {panel.shape[1]} columns")
panel.columns.tolist()

## The formation price

One row per `(symbol, formation_year)`: the last session on or before 30 June of that year.
`price_close * unit` is the **raw** quote, the same convention the BM notebook uses.

`adj_ratio` rides along, and `close_adj = close_raw / adj_ratio` is the split- and
dividend-adjusted series — FireAnt's ratio adjusts for cash dividends too, so a return
computed from `close_adj` is a total return. It is cumulative to the crawl date, which makes
it internally consistent across this whole panel but not comparable to a series fetched
later. Use `close_raw` for anything that has to be the price on the day, `close_adj` for
anything that spans two dates.

Nothing is dropped for staleness here: `price_lag_days` and `price_traded` are exposed and
`tradeable` is left to be the single gate. It already is one — of the panel rows whose last
print is more than 30 days before formation (73 of them), **none** are `tradeable`, and every
one of the 11,445 tradeable rows is priced within 13 days.

In [ ]:
AS_OF_PRICE_SQL = f"""
select symbol,
       formation_year,
       date                as price_date,
       price_close * unit  as close_raw,
       adj_ratio,
       total_volume > 0    as price_traded,
       cast(julianday(substr(date, 1, 4) || '-{FORMATION_MMDD}')
            - julianday(date) as integer) as price_lag_days
from (
    select symbol,
           cast(substr(date, 1, 4) as integer) as formation_year,
           date, price_close, unit, adj_ratio, total_volume,
           row_number() over (partition by symbol, substr(date, 1, 4)
                              order by date desc) as rn
    from fireant_prices
    where unit = 1000
      and price_close > 0
      and date <= substr(date, 1, 4) || '-{FORMATION_MMDD}'
)
where rn = 1
"""

with closing(sqlite3.connect(DB)) as conn:
    formation_price = pd.read_sql(AS_OF_PRICE_SQL, conn)

formation_price["close_adj"] = formation_price["close_raw"] / formation_price["adj_ratio"]
formation_price["price_traded"] = formation_price["price_traded"].astype(bool)
formation_price["period"] = formation_price["formation_year"] - FORMATION_LAG_YEARS
formation_price = (formation_price
                   .drop(columns="formation_year")
                   .set_index(["symbol", "period"])
                   .sort_index())
formation_price

In [ ]:
# How far the matched print sits from 30 June. Anything past a long weekend is a stock that
# stopped printing before formation day, not a calendar effect.
lag = formation_price["price_lag_days"]

pd.Series({
    "symbol-years priced": len(formation_price),
    "exactly on the formation date": int(lag.eq(0).sum()),
    "  within 2 days (weekend)": int(lag.le(2).sum()),
    "  3 to 30 days": int(lag.between(3, 30).sum()),
    "  over 30 days": int(lag.gt(30).sum()),
    "longest gap, days": int(lag.max()),
    "priced off a session with no trade": int((~formation_price["price_traded"]).sum()),
})

In [ ]:
panel = panel.join(formation_price[["price_date", "price_lag_days", "price_traded",
                                    "close_raw", "adj_ratio", "close_adj"]])

# The turnover notebook's `last_bar` is the last bar inside the 30-day window; the price
# match looks back further when it has to. Where the stock traded in the window at all the
# two must be the same session, and that is the check that the two notebooks are matching
# on the same calendar.
same_session = panel["last_bar"].notna() & panel["price_date"].notna()
print(f"rows where both notebooks found a bar: {int(same_session.sum())}, "
      f"disagreeing on the session: {int((panel.loc[same_session, 'last_bar'] != panel.loc[same_session, 'price_date']).sum())}")

panel[["f_score", "bm", "turnover", "tradeable", "price_date", "close_raw", "close_adj"]]

## Coverage of the joined panel

Each block can be missing for its own reason, and they do not overlap neatly: an F-score
needs three consecutive clean years, a BM needs a December close and positive book equity, a
turnover needs bars in the June window. The row that matters is the last one — everything
present, and tradeable.

In [ ]:
have = pd.DataFrame({
    "f_score": panel["f_score"].notna(),
    "bm": panel["bm"].notna(),
    "turnover": panel["turnover"].notna(),
    "price": panel["close_raw"].notna(),
})
complete = have.all(axis=1)
investable = complete & panel["tradeable"].fillna(False)

pd.Series({
    "panel rows": len(panel),
    "with f_score": int(have["f_score"].sum()),
    "with bm": int(have["bm"].sum()),
    "with turnover": int(have["turnover"].sum()),
    "with a formation price": int(have["price"].sum()),
    "complete (all four)": int(complete.sum()),
    "complete and tradeable": int(investable.sum()),
    f"  and f_score >= {CUTOFF}": int((investable & panel["f_score"].ge(CUTOFF)).sum()),
})

In [ ]:
# The same by fiscal period, with the formation year each row is priced at. `investable` is
# the universe a backtest actually ranks; `selected` is what the strategy buys out of it.
pd.DataFrame({
    "formation": panel.groupby(level="period")["formation_year"].first(),
    "rows": panel.groupby(level="period").size(),
    "complete": complete.groupby(level="period").sum(),
    "investable": investable.groupby(level="period").sum(),
    "selected": (investable & panel["f_score"].ge(CUTOFF)).groupby(level="period").sum(),
    "median_bm": panel.loc[investable, "bm"].groupby(level="period").median().round(3),
    "median_turnover": panel.loc[investable, "turnover"].groupby(level="period").median().round(4),
})

In [ ]:
# What each screen costs, in order, on the rows that have an F-score at all. Read down: this
# is the funnel from "scored" to "bought".
scored = have["f_score"]
funnel = pd.Series({
    "scored": int(scored.sum()),
    "+ has bm": int((scored & have["bm"]).sum()),
    "+ has a formation price": int((scored & have["bm"] & have["price"]).sum()),
    "+ tradeable": int(investable.sum()),
    f"+ f_score >= {CUTOFF}": int((investable & panel["f_score"].ge(CUTOFF)).sum()),
})
funnel.to_frame("rows").assign(pct_of_scored=lambda d: (100 * d["rows"] / funnel.iloc[0]).round(1))

## Exit price and forward return

The position opens at `close_adj` on 30 June F and closes at `close_adj` on 30 June
F + `HOLDING_YEARS`. Both ends adjusted, always — a raw entry against an adjusted exit, or
either end raw, books every dividend and every bonus issue as a capital loss.

**The exit is looked up as-of, not by shifting the panel.** The panel's rows are gated by
the accounting checks, so a firm whose *next* year's statements failed them loses its row —
while the stock kept trading and the return exists. Shifting drops those, and they are not a
random sample: on this panel they run about 7pp below the rest, so dropping them flatters
the result.

**`exit_lag_days` is the whole delisting story.** There is no delisting flag anywhere in
`fireant_prices`; the only observable is that a symbol stops printing. The gap between the
exit date and the last bar before it is the evidence, and it splits three ways: a long
weekend, a thin stretch, or a stock that left. `exit_permanent` marks the last case — no
print for over `MAX_EXIT_LAG_DAYS` before the exit *and* none after it either. A stock that
merely paused and came back is not marked, because it did not leave.

**`DELISTING_RETURN` is left at `None` on purpose.** The default exits at the last traded
price, which invents nothing, but it is optimistic: some of those positions come out at
exactly the entry price, and a stock thrown off the exchange rarely leaves a holder whole.
The sensitivity table below is the answer to that, rather than a haircut picked to look
defensible. `-1.0` would be the wrong default in this market: a ticker that moves from HOSE
to UPCoM, merges, or simply leaves the 1,830-symbol crawl universe looks identical to a
bankruptcy, and booking it at -100% fabricates a loss that never happened.

In [ ]:
# The as-of lookup only needs month-end bars, because the exit date is the last calendar day
# of a month: the last bar on or before it is the last bar of that month, or of the most
# recent earlier month that has one. If FORMATION_MMDD ever moves off a month end this
# reduction stops being valid and the merge has to run on daily bars.
MONTH_END_SQL = """
select symbol, date, price_close * unit / adj_ratio as close_adj
from (
    select symbol, date, price_close, unit, adj_ratio,
           row_number() over (partition by symbol, substr(date, 1, 7)
                              order by date desc) as rn
    from fireant_prices
    where unit = 1000 and price_close > 0
)
where rn = 1
"""

with closing(sqlite3.connect(DB)) as conn:
    month_end = pd.read_sql(MONTH_END_SQL, conn, parse_dates=["date"])

month_end = month_end.sort_values("date")
data_end = month_end["date"].max()
last_print = month_end.groupby("symbol")["date"].max()

print(f"{len(month_end)} month-end bars, {month_end['symbol'].nunique()} symbols, "
      f"price data ends {data_end.date()}")

In [ ]:
period = panel.index.get_level_values("period")
exit_year = period + FORMATION_LAG_YEARS + HOLDING_YEARS

targets = pd.DataFrame({
    "symbol": panel.index.get_level_values("symbol"),
    "period": period,
    "exit_date": pd.to_datetime(exit_year.astype(str) + f"-{FORMATION_MMDD}"),
}).sort_values("exit_date")

exit_px = (pd.merge_asof(targets, month_end, left_on="exit_date", right_on="date",
                         by="symbol", direction="backward")
           .set_index(["symbol", "period"])
           .reindex(panel.index))

panel["exit_date"] = exit_px["exit_date"]
panel["exit_price_date"] = exit_px["date"]
panel["close_adj_exit"] = exit_px["close_adj"]
panel["exit_lag_days"] = (exit_px["exit_date"] - exit_px["date"]).dt.days

# The holding period has to have finished. Left alone, merge_asof prices a 2027 exit off the
# last bar in the table, which is not an exit — it is an open position, and counting it would
# put a partial year into the cross-section as if it were a full one.
panel["holding_complete"] = panel["exit_date"].le(data_end)

# Permanently gone: nothing printed for MAX_EXIT_LAG_DAYS before the exit, and nothing after
# it either. The second half is what separates a delisting from a long suspension.
resumed = pd.Series(last_print.reindex(panel.index.get_level_values("symbol")).to_numpy(),
                    index=panel.index).gt(panel["exit_date"])
panel["exit_permanent"] = (panel["holding_complete"]
                           & panel["exit_lag_days"].gt(MAX_EXIT_LAG_DAYS)
                           & ~resumed)

panel[["price_date", "close_adj", "exit_date", "exit_price_date", "close_adj_exit",
       "exit_lag_days", "holding_complete", "exit_permanent"]]

In [ ]:
# How the exit prints sit against the exit date, on finished holding periods only.
finished = investable & panel["holding_complete"]
exit_lag = panel.loc[finished, "exit_lag_days"]

pd.Series({
    "investable, holding period finished": int(finished.sum()),
    "  no exit price at all": int((finished & panel["close_adj_exit"].isna()).sum()),
    "exit lag <= 3 days (weekend)": int(exit_lag.le(3).sum()),
    f"  4 to {MAX_EXIT_LAG_DAYS} days": int(exit_lag.between(4, MAX_EXIT_LAG_DAYS).sum()),
    f"  over {MAX_EXIT_LAG_DAYS} days": int(exit_lag.gt(MAX_EXIT_LAG_DAYS).sum()),
    "    traded again later (paused)": int((finished & exit_lag.gt(MAX_EXIT_LAG_DAYS)
                                            & ~panel["exit_permanent"]).sum()),
    "    never traded again (left)": int((finished & panel["exit_permanent"]).sum()),
    "investable, still open (no exit yet)": int((investable & ~panel["holding_complete"]).sum()),
})

In [ ]:
gross_return = panel["close_adj_exit"] / panel["close_adj"] - 1
panel["fwd_return_1y"] = gross_return.where(panel["holding_complete"])

if DELISTING_RETURN is not None:
    panel.loc[panel["exit_permanent"], "fwd_return_1y"] = DELISTING_RETURN

selected = finished & panel["f_score"].ge(CUTOFF)
pd.DataFrame({
    "n": [int(finished.sum()), int(selected.sum())],
    "median": [panel.loc[finished, "fwd_return_1y"].median(),
               panel.loc[selected, "fwd_return_1y"].median()],
    "mean": [panel.loc[finished, "fwd_return_1y"].mean(),
             panel.loc[selected, "fwd_return_1y"].mean()],
}, index=["investable", f"selected (F >= {CUTOFF})"]).round(4)

In [ ]:
# What the delisting policy is worth. It bites on the `exit_permanent` rows only, and those
# are a fraction of a percent of the panel — the point of the table is that the conclusion
# does not depend on which number goes into DELISTING_RETURN, so none of them has to be
# argued for. Report it and the question is closed.
def with_policy(delisting_return):
    r = gross_return.where(panel["holding_complete"])
    if delisting_return is not None:
        r = r.mask(panel["exit_permanent"], delisting_return)
    return r


pd.DataFrame([{
    "policy": "last traded price" if dr is None else f"{dr:.0%}",
    "investable median": round(with_policy(dr)[finished].median(), 4),
    "investable mean": round(with_policy(dr)[finished].mean(), 4),
    f"F>={CUTOFF} median": round(with_policy(dr)[selected].median(), 4),
    f"F>={CUTOFF} mean": round(with_policy(dr)[selected].mean(), 4),
} for dr in [None, -0.30, -0.50, -1.00]]).set_index("policy")

In [ ]:
# Forward return by fiscal period. The last period is empty by construction: its holding year
# has not finished inside the price data.
pd.DataFrame({
    "investable": finished.groupby(level="period").sum(),
    "inv_median": panel.loc[finished, "fwd_return_1y"].groupby(level="period").median().round(4),
    "selected": selected.groupby(level="period").sum(),
    "sel_median": panel.loc[selected, "fwd_return_1y"].groupby(level="period").median().round(4),
    "gone": (finished & panel["exit_permanent"]).groupby(level="period").sum(),
})

In [ ]:
# Spot check: one firm, four years, end to end.
panel.loc["HPG", ["f_score", "bm", "turnover", "tradeable", "price_date", "close_adj",
                  "exit_price_date", "close_adj_exit", "fwd_return_1y"]].tail(4)

### Export

`final_panel.csv`, keyed on `(symbol, period)` — one row per firm-year, carrying the score,
the valuation, the liquidity verdict, the price it would be bought at, the price it would be
sold at, and the return between them. Downstream this is the only file the backtest has to
read.

Read `fwd_return_1y` together with `holding_complete` and `exit_permanent`: the first says
whether the holding year finished at all, the second whether the exit price is a real exit
or the last print of a stock that left.

In [ ]:
panel.to_csv(f"{RESULTS}/final_panel.csv")
panel.shape